# 10 - 結果總覽與分析

彙整所有實驗結果，產出報告所需的圖表：

| Fig | 內容 |
|-----|------|
| 15 | 全模型比較表 (6 models × acc/F1) |
| 16 | In-corpus confusion matrix heatmap (CNN/LSTM/wav2vec) |
| 17 | Per-class F1 比較 + human baseline ~65% |
| 18 | Cross-corpus 各情緒 F1 退化 |
| 19 | wav2vec LOCO confusion matrix (4 corpus) |
| 20 | Train pool size vs accuracy (CREMA-D confound) |

In [ ]:
# === Setup ===
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

PROJECT_ROOT = Path('..').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
LOCO_DIR = RESULTS_DIR / 'cross_corpus'
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)


# === 載入所有結果 ===
baseline = load_json(RESULTS_DIR / 'baseline_ml' / 'baseline_results.json')
cnn = load_json(RESULTS_DIR / 'cnn_results.json')
lstm = load_json(RESULTS_DIR / 'lstm_results.json')
wav2vec = load_json(RESULTS_DIR / 'wav2vec_results.json')
loco_cnn = load_json(LOCO_DIR / 'loco_cnn.json')
loco_lstm = load_json(LOCO_DIR / 'loco_lstm.json')
loco_wav2vec = load_json(LOCO_DIR / 'loco_wav2vec.json')

CLASSES = cnn.get('classes', ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad'])
DATASETS = ['RAVDESS', 'CREMA-D', 'TESS', 'SAVEE']

print('All results loaded.')
print(f'Classes: {CLASSES}')

In [ ]:
# === Fig 15: 全模型比較表 ===

rows = []
# Baseline ML
for model_name in ['SVM', 'RandomForest', 'XGBoost']:
    m = baseline[model_name]
    folds = m['folds']
    accs = [f['accuracy'] for f in folds]
    f1ws = [f['f1_weighted'] for f in folds]
    f1ms = [f['f1_macro'] for f in folds]
    rows.append({
        'Model': model_name,
        'Feature': 'MFCC stats (234D)',
        'Accuracy': f'{np.mean(accs):.4f} ± {np.std(accs):.4f}',
        'F1 (weighted)': f'{np.mean(f1ws):.4f} ± {np.std(f1ws):.4f}',
        'F1 (macro)': f'{np.mean(f1ms):.4f} ± {np.std(f1ms):.4f}',
    })

# DL models
for model_name, data, feat in [
    ('CNN', cnn, 'Mel-spectrogram (128×94)'),
    ('Bi-LSTM+Attn', lstm, 'MFCC sequence (94×39)'),
    ('wav2vec2', wav2vec, 'Pre-trained (768D)'),
]:
    s = data['summary']
    rows.append({
        'Model': model_name,
        'Feature': feat,
        'Accuracy': f'{s["accuracy_mean"]:.4f} ± {s["accuracy_std"]:.4f}',
        'F1 (weighted)': f'{s["f1_weighted_mean"]:.4f} ± {s["f1_weighted_std"]:.4f}',
        'F1 (macro)': f'{s["f1_macro_mean"]:.4f} ± {s["f1_macro_std"]:.4f}',
    })

comp_df = pd.DataFrame(rows)
print('Table 1: In-corpus Model Comparison (5-Fold CV)')
print(comp_df.to_string(index=False))

# Plotly table
fig = go.Figure(data=[go.Table(
    header=dict(values=list(comp_df.columns), fill_color='steelblue',
                font=dict(color='white', size=13), align='center'),
    cells=dict(values=[comp_df[c] for c in comp_df.columns],
               fill_color=[['#f0f0f0', 'white'] * 3],
               align='center', height=30),
)])
fig.update_layout(title='In-corpus Model Comparison (5-Fold CV, StratifiedGroupKFold)',
                  height=350, width=900)
fig.write_html(FIG_DIR / '15_model_comparison_table.html')
fig.write_image(FIG_DIR / '15_model_comparison_table.png', width=900, height=350, scale=2)
fig.show()

In [ ]:
# === Fig 16: In-corpus Confusion Matrix Heatmap (3 panels) ===
# 5-fold 的 confusion matrix 加總後 row-normalize

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['CNN', 'Bi-LSTM+Attn', 'wav2vec2'],
    horizontal_spacing=0.08,
)

for col_idx, (model_name, data) in enumerate(
    [('CNN', cnn), ('LSTM', lstm), ('wav2vec', wav2vec)], 1
):
    # 加總 5-fold confusion matrices
    cm_sum = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for cm in data['confusion_matrices']:
        cm_sum += np.array(cm)

    # Row-normalize → recall percentage
    cm_norm = cm_sum.astype(float) / cm_sum.sum(axis=1, keepdims=True) * 100

    # 標註文字：百分比 + 原始數字
    text = [[f'{cm_norm[i][j]:.1f}%\n({cm_sum[i][j]})'
             for j in range(len(CLASSES))] for i in range(len(CLASSES))]

    fig.add_trace(
        go.Heatmap(
            z=cm_norm, x=CLASSES, y=CLASSES,
            text=text, texttemplate='%{text}', textfont=dict(size=9),
            colorscale='Blues', showscale=(col_idx == 3),
            zmin=0, zmax=100,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title='In-corpus Confusion Matrix (5-Fold Aggregated, Row-Normalized %)',
    height=500, width=1500, template='plotly_white',
)
for i in range(1, 4):
    fig.update_xaxes(title_text='Predicted', row=1, col=i)
    fig.update_yaxes(title_text='True' if i == 1 else '', row=1, col=i, autorange='reversed')

fig.write_html(FIG_DIR / '16_incorpus_confusion_matrices.html')
fig.write_image(FIG_DIR / '16_incorpus_confusion_matrices.png', width=1500, height=500, scale=2)
fig.show()

In [ ]:
# === Fig 17: Per-class F1 比較 + Human Baseline ===
# 從 classification_reports 取 per-class F1 (5-fold 平均)

f1_data = {}
for model_name, data in [('CNN', cnn), ('LSTM', lstm), ('wav2vec2', wav2vec)]:
    per_class_f1 = {emo: [] for emo in CLASSES}
    for report in data['classification_reports']:
        for emo in CLASSES:
            per_class_f1[emo].append(report[emo]['f1-score'])
    f1_data[model_name] = {emo: np.mean(vals) for emo, vals in per_class_f1.items()}

# 繪圖
colors = {'CNN': '#3498db', 'LSTM': '#e74c3c', 'wav2vec2': '#2ecc71'}

fig = go.Figure()
for model_name, f1s in f1_data.items():
    fig.add_trace(go.Bar(
        x=CLASSES, y=[f1s[emo] for emo in CLASSES],
        name=model_name, marker_color=colors[model_name],
        text=[f'{f1s[emo]:.3f}' for emo in CLASSES], textposition='outside',
    ))

# Human baseline 虛線
fig.add_hline(
    y=0.65, line_dash='dash', line_color='gray', line_width=2,
    annotation_text='Human agreement ~65%',
    annotation_position='top right',
    annotation_font=dict(size=12, color='gray'),
)

fig.update_layout(
    title='Per-class F1 Score Comparison (In-corpus, 5-Fold Mean)',
    barmode='group',
    yaxis_title='F1 Score', yaxis_range=[0, 1],
    xaxis_title='Emotion',
    height=500, width=900, template='plotly_white',
)

fig.write_html(FIG_DIR / '17_perclass_f1_comparison.html')
fig.write_image(FIG_DIR / '17_perclass_f1_comparison.png', width=900, height=500, scale=2)
fig.show()

In [ ]:
# === Fig 18: Cross-corpus 各情緒 F1 退化 ===
# LOCO 4 輪平均 per-class F1 vs in-corpus per-class F1

loco_data = {'CNN': loco_cnn, 'LSTM': loco_lstm, 'wav2vec2': loco_wav2vec}

# LOCO per-class F1 (4 corpus 平均)
loco_f1 = {}
for model_name, loco in loco_data.items():
    per_class = {emo: [] for emo in CLASSES}
    for ds in DATASETS:
        key = f'test_{ds}'
        if key in loco:
            report = loco[key]['classification_report']
            for emo in CLASSES:
                per_class[emo].append(report[emo]['f1-score'])
    loco_f1[model_name] = {emo: np.mean(vals) for emo, vals in per_class.items()}

# 繪製: in-corpus vs cross-corpus per emotion
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['CNN', 'LSTM', 'wav2vec2'],
    horizontal_spacing=0.08,
)

for col_idx, model_name in enumerate(['CNN', 'LSTM', 'wav2vec2'], 1):
    ic_vals = [f1_data[model_name][emo] for emo in CLASSES]
    cc_vals = [loco_f1[model_name][emo] for emo in CLASSES]

    fig.add_trace(
        go.Bar(x=CLASSES, y=ic_vals, name='In-corpus',
               marker_color='steelblue', showlegend=(col_idx == 1),
               legendgroup='in'),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Bar(x=CLASSES, y=cc_vals, name='Cross-corpus (LOCO)',
               marker_color='salmon', showlegend=(col_idx == 1),
               legendgroup='cross'),
        row=1, col=col_idx,
    )
    fig.update_yaxes(range=[0, 1], row=1, col=col_idx)

fig.update_layout(
    title='Per-Emotion F1 Degradation: In-corpus vs Cross-corpus (LOCO Mean)',
    barmode='group',
    height=500, width=1400, template='plotly_white',
)

fig.write_html(FIG_DIR / '18_crosscorpus_degradation_by_emotion.html')
fig.write_image(FIG_DIR / '18_crosscorpus_degradation_by_emotion.png', width=1400, height=500, scale=2)
fig.show()

In [ ]:
# === Fig 19: wav2vec LOCO Confusion Matrix (4 panels) ===

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f'Test: {ds}' for ds in DATASETS],
    horizontal_spacing=0.06,
)

for col_idx, ds in enumerate(DATASETS, 1):
    key = f'test_{ds}'
    cm = np.array(loco_wav2vec[key]['confusion_matrix'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    text = [[f'{cm_norm[i][j]:.0f}%' for j in range(len(CLASSES))]
            for i in range(len(CLASSES))]

    fig.add_trace(
        go.Heatmap(
            z=cm_norm, x=CLASSES, y=CLASSES,
            text=text, texttemplate='%{text}', textfont=dict(size=8),
            colorscale='Blues', showscale=(col_idx == 4),
            zmin=0, zmax=100,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title='wav2vec2 LOCO Confusion Matrix (Row-Normalized %, per Test Corpus)',
    height=450, width=1600, template='plotly_white',
)
for i in range(1, 5):
    fig.update_yaxes(autorange='reversed', row=1, col=i)

fig.write_html(FIG_DIR / '19_wav2vec_loco_confusion_matrices.html')
fig.write_image(FIG_DIR / '19_wav2vec_loco_confusion_matrices.png', width=1600, height=450, scale=2)
fig.show()

In [ ]:
# === Fig 20: Train Pool Size vs Accuracy (CREMA-D Confound) ===

# 計算每輪 LOCO 的 train pool 大小
meta_df = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
pool_sizes = {}
for ds in DATASETS:
    pool_sizes[ds] = len(meta_df[meta_df['dataset'] != ds])

print('Train pool sizes when each corpus is test:')
for ds, size in pool_sizes.items():
    print(f'  Test={ds}: train pool = {size:,}')

# 繪圖
model_colors = {'CNN': '#3498db', 'LSTM': '#e74c3c', 'wav2vec2': '#2ecc71'}

fig = go.Figure()
for model_name, loco in loco_data.items():
    x_vals = [pool_sizes[ds] for ds in DATASETS]
    y_vals = [loco[f'test_{ds}']['accuracy'] for ds in DATASETS]
    labels = DATASETS

    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='markers+text',
        marker=dict(size=14, color=model_colors[model_name]),
        text=labels, textposition='top center', textfont=dict(size=10),
        name=model_name,
    ))

fig.update_layout(
    title='LOCO Accuracy vs Train Pool Size (CREMA-D Confounding Factor)',
    xaxis_title='Train Pool Size (samples)',
    yaxis_title='Test Accuracy',
    yaxis_range=[0, 0.8],
    height=500, width=800, template='plotly_white',
)

# 標註 CREMA-D 區域
fig.add_annotation(
    x=pool_sizes['CREMA-D'], y=0.05,
    text='← CREMA-D as test:\ntrain pool only 3,876',
    showarrow=False, font=dict(size=11, color='red'),
)

fig.write_html(FIG_DIR / '20_train_pool_size_analysis.html')
fig.write_image(FIG_DIR / '20_train_pool_size_analysis.png', width=800, height=500, scale=2)
fig.show()

In [ ]:
# === Silhouette Score 對照表 ===

sil_path = PROJECT_ROOT / 'data' / 'embeddings' / 'silhouette_scores.json'
if sil_path.exists():
    with open(sil_path) as f:
        sil_scores = json.load(f)

    sil_df = pd.DataFrame([
        {'Model': 'CNN', 'Silhouette': sil_scores.get('cnn', 'N/A'),
         'In-corpus Acc': cnn['summary']['accuracy_mean']},
        {'Model': 'LSTM', 'Silhouette': sil_scores.get('lstm', 'N/A'),
         'In-corpus Acc': lstm['summary']['accuracy_mean']},
        {'Model': 'wav2vec2', 'Silhouette': sil_scores.get('wav2vec', 'N/A'),
         'In-corpus Acc': wav2vec['summary']['accuracy_mean']},
    ])
    print('Silhouette Score vs In-corpus Accuracy:')
    print(sil_df.to_string(index=False))
else:
    print('Silhouette scores not found — run NB09 first')

In [ ]:
# === 統計顯著性（Paired t-test on 5-fold metrics）===

pairs = [
    ('CNN', 'LSTM', cnn, lstm),
    ('CNN', 'wav2vec2', cnn, wav2vec),
    ('LSTM', 'wav2vec2', lstm, wav2vec),
]

print('Paired t-test on 5-fold accuracy (two-tailed):')
print(f'{"Pair":<25s} {"t-stat":>8s} {"p-value":>10s} {"Significant?":>14s}')
print('-' * 60)

for name_a, name_b, data_a, data_b in pairs:
    accs_a = [f['accuracy'] for f in data_a['folds']]
    accs_b = [f['accuracy'] for f in data_b['folds']]
    t_stat, p_val = stats.ttest_rel(accs_a, accs_b)
    sig = 'Yes (p<0.05)' if p_val < 0.05 else 'No'
    print(f'{name_a} vs {name_b:<14s} {t_stat:>8.3f} {p_val:>10.4f} {sig:>14s}')

## 總結

### 核心發現

1. **SVM ≈ LSTM+Attention** — 同樣使用 MFCC 特徵，模型複雜度的提升幾乎無效
2. **CNN (Mel-spec) > LSTM (MFCC)** — 換特徵比換模型架構更有效 (+10%)
3. **wav2vec2 >> CNN** — Pre-trained representation 帶來壓倒性優勢 (+15%)
4. **Cross-corpus: wav2vec 遷移力遠優於 CNN/LSTM** — 52.2% vs ~28%
5. **CREMA-D anomaly** — 需考慮 train pool size confounding factor

### 故事線

**「模型架構不是瓶頸，特徵表徵才是。」**